# Data Preparation Notebook 1: NCI thesaurus (NCIt)
For each concept code in the NCI `Thesaurus.txt` file, get all definitions and synonyms. This information will be used to build `variable_para` and `alternate_variable_para`, interoperable rows to use for embedding model training on standard data elements 
This notebook operates on `Thesaurus.txt` file and outputs the following two files:
- definitions.csv   columns: concept, definition, source
- synonyms.csv       columns: concept, name, term_type, source
- Each concept in NCIt file can have multiple definitions and multiple synonyms, each tagged with its own source (NCI, CDISC, CDISC-GLOSS, caDSR, CDC, etc). For
example, NCIt code C17049 ('Race') has 4 separate definition rows (NCI,
CDISC-GLOSS, CDISC, OORO) -- so it will produce 4 rows in definitions.csv, one
per definition
- In Notebook 2, we will groups rows back by `concept` to build `variable_para` and `alt_variable_para` per concept
- `Thesaurus.txt` file is downloaded from NCI EVS link, see here https://evs.nci.nih.gov/ftp1/NCI_Thesaurus/Thesaurus_26.06e.FLAT.zip. It has 212234 rows.
- Added a header to this file, following description in `Readme.txt`.See link https://evs.nci.nih.gov/ftp1/NCI_Thesaurus/ReadMe.txt
- Note: this script takes 1 day to run. Run it locally

### USAGE
pip install requests \
python3 harvest_ncit_definitions.py /path/to/Thesaurus.txt --definitions-out definitions.csv --synonyms-out synonyms.csv

In [ ]:
# imports

import argparse
import csv
import json
import os
import sys
import time
import requests
import pandas as pd

In [ ]:
# NCIt API URL
NCIt_URL = "https://api-evsrest.nci.nih.gov/api/v1"

#### Get list of unique concept codes from `Thesaurus.txt`

In [ ]:
thesaurus_data = pd.read_csv('Thesaurus_26.06e.txt', sep='\t')

In [ ]:
thesaurus_data.shape

In [ ]:
thesaurus_data.columns

In [ ]:
codes = sorted(thesaurus_data['code'].unique())

In [ ]:
len(codes)

In [ ]:
codes[:10]

In [ ]:
# create EvsApiClient class with retries, timeout, rate_per_sec to query the API
# cache already queried codes onto a cache dir
# make new requests by checking cache dir first
# get_concept returns a JSON dictionary for concept `code`, or None if not found
class EvsApiClient:
    def __init__(self, cache_dir=None, rate_per_sec=10, max_retries=3, timeout=20):
        self.cache_dir = cache_dir
        if cache_dir:
            os.makedirs(cache_dir, exist_ok=True)
        self.min_interval = 1.0 / rate_per_sec if rate_per_sec > 0 else 0
        self.max_retries = max_retries
        self.timeout = timeout
        self._last_request_time = 0.0

    def _cache_path(self, code):
        if not self.cache_dir:
            return None
        return os.path.join(self.cache_dir, f"{code}.json")

    def get_concept(self, code):
        cache_path = self._cache_path(code)
        if cache_path and os.path.exists(cache_path):
            try:
                with open(cache_path, encoding="utf-8") as f:
                    return json.load(f)
            except (json.JSONDecodeError, OSError):
                pass

        elapsed = time.time() - self._last_request_time
        # wait for a minimum interval to avoid bombarding NCIT server
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)

        url = f"{NCIt_URL}/concept/ncit/{code}"
        last_error = None
        for attempt in range(1, self.max_retries + 1):
            try:
                self._last_request_time = time.time()
                resp = requests.get(url, params={"include": "full"}, timeout=self.timeout)
                if resp.status_code == 404:
                    return None
                resp.raise_for_status()
                data = resp.json()
                if cache_path:
                    with open(cache_path, "w", encoding="utf-8") as f:
                        json.dump(data, f)
                return data
            except Exception as e:
                last_error = e
                time.sleep(min(2 ** attempt, 10))

        print(f"WARNING: failed to fetch concept {code} after {self.max_retries} attempts ({last_error}).",
              file=sys.stderr)
        return None
    

In [ ]:
# track progress
class ProgressTracker:
    def __init__(self, path):
        self.path = path
        self.done_codes = set()
        if os.path.exists(path):
            try:
                with open(path, encoding="utf-8") as f:
                    data = json.load(f)
                self.done_codes = set(data.get("done_codes", []))
            except (json.JSONDecodeError, OSError):
                print(f"WARNING: could not read progress file {path}; starting fresh.", file=sys.stderr)

    def is_done(self, code):
        return code in self.done_codes

    def mark_done(self, code):
        self.done_codes.add(code)
        self._save()

    def _save(self):
        tmp_path = self.path + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump({"done_codes": sorted(self.done_codes)}, f)
        os.replace(tmp_path, self.path)

### Format rows to write in output files
- definitions and synonyms files

In [ ]:
def definition_rows_for_code(code, concept_json):
    """one row per definition entry: we will output the following columns:
    concept, definition, source
    e.g.
    "C100000","A percutaneous coronary intervention is necessary for a myocardial infarction that presents with ST segment elevation and the subject does not have recurrent or persistent symptoms, symptoms of heart failure or ventricular arrhythmia. The presentation is past twelve hours since onset of symptoms.","CDISC"
    "C100000","A percutaneous coronary intervention is necessary for a myocardial infarction that presents with ST segment elevation and the subject does not have recurrent or persistent symptoms, symptoms of heart failure or ventricular arrhythmia. The presentation is past twelve hours since onset of symptoms. (ACC)","NCI"
    """
    rows = []
    for d in (concept_json or {}).get("definitions") or []:
        rows.append([code, d.get("definition", ""), d.get("source", "")])
    return rows

In [ ]:
def synonym_rows_for_code(code, concept_json):
    """One row per synonym entry: we will output the following columns:
    concept, name, term_type, source
    term type: see https://evsexplore.semantics.cancer.gov/evsexplore/termtypes/ncit
    """
    rows = []
    for s in (concept_json or {}).get("synonyms") or []:
        rows.append([code, s.get("name", ""), s.get("termType", ""), s.get("source", "")])
    return rows

### define output files and variables

In [ ]:
definitions_out_file = 'definitions.ncit.csv'
synonyms_out_file = 'synonyms.ncit.csv'
progress_file = 'harvest_progress.ncit.json'
cache_dir = '.ncit.evs.cache'
# max API requests per sec
rate = 10.0

### Run data prep

In [ ]:
print(f'Processing {len(codes)} unique concept codes from Thesaurus.txt file')

In [ ]:
api_client = EvsApiClient(cache_dir=cache_dir, rate_per_sec=rate)
progress = ProgressTracker(progress_file)

In [ ]:
# check how many codes already processed/how many left
already_done = sum(1 for c in codes if progress.is_done(c))
if already_done:
    print(
        f"Resuming: {already_done:,} of {len(codes):,} code(s) already processed in a previous run; "
        f"skipping those."
    )
est_remaining = len(codes) - already_done
est_hours = (est_remaining / rate) / 3600 if rate > 0 else 0
print(
    f"{est_remaining:,} code(s) remaining. Rough estimate at {rate}/sec: ~{est_hours:.1f} hours "
    f"(less if many are already in --cache-dir from a prior run). Interrupt anytime with Ctrl+C and "
    f"re-run the same command to resume."
)

In [ ]:
# create output files if they dont exist and add header rows if they dont exist
def_write_header = not os.path.exists(definitions_out_file)
syn_write_header = not os.path.exists(synonyms_out_file)
def_file = open(definitions_out_file, "a", newline="", encoding="utf-8")
syn_file = open(synonyms_out_file, "a", newline="", encoding="utf-8")
def_writer = csv.writer(def_file, quoting=csv.QUOTE_ALL)
syn_writer = csv.writer(syn_file, quoting=csv.QUOTE_ALL)
if def_write_header:
    def_writer.writerow(["concept", "definition", "source"])
    def_file.flush()
if syn_write_header:
    syn_writer.writerow(["concept", "name", "term_type", "source"])
    syn_file.flush()

In [ ]:
processed_this_run = 0
def_rows_written = 0
syn_rows_written = 0
try:
    # count codes from 1
    for i, code in enumerate(codes, 1):
            if progress.is_done(code):
                continue
            if i % 500 == 0 or i == 1:
                print(f"[{i}/{len(codes)}] ({code}) ...")

            concept_json = api_client.get_concept(code)
            if concept_json is None:
                progress.mark_done(code)  # don't retry a genuine 404 forever
                continue

            for row in definition_rows_for_code(code, concept_json):
                def_writer.writerow(row)
                def_rows_written += 1
            for row in synonym_rows_for_code(code, concept_json):
                syn_writer.writerow(row)
                syn_rows_written += 1
            def_file.flush()
            syn_file.flush()

            progress.mark_done(code)
            processed_this_run += 1
except KeyboardInterrupt:
    print(
        f"\nInterrupted by user. {processed_this_run:,} new code(s) processed this run; progress saved. "
        f"Re-run the same command to resume."
    )
finally:
    def_file.close()
    syn_file.close()

print(
    f"Done. Processed {processed_this_run:,} new code(s) this run "
    f"({def_rows_written:,} definition rows, {syn_rows_written:,} synonym rows written)."
)